# Medical MAE Backbone Verification

**Model**: Medical MAE (`lambert-x/medical_mae`) — MAE pre-trained ViT on CXR data  
**Arch**: ViT-Base/16 (86M) or ViT-Small/16 (22M)  
**Embed dim**: 768 (Base) / 384 (Small) — CLS token (default) or global avg pool  
**Input**: 3×224×224 px, chestx-ray normalization (`[0.5056]*3 / [0.252]*3`)  
**Access**: Weights on Google Drive (see README in repo)  
**Zero-shot**: No — vision-only MAE pretraining

Paper: [Delving into Masked Autoencoders for Multi-Label Thorax Disease Classification (WACV 2023)](https://arxiv.org/abs/2210.12843)

Pre-trained on 0.3M CXR images (ViT-Small) or 0.5M (ViT-Base) from NIH/CheXpert/MIMIC.

### Prerequisites

The repo and a legacy `timm==0.4.12` are auto-installed on first run.
You only need to download the pretrained weights manually:

- ViT-Small: `vit_small_patch16_CXR_0.3M_mae_pretrain.pth`
- ViT-Base:  `vit-b_CXR_0.5M_mae.pth`

Place under `third_party_models/medical_mae/`.


In [ ]:
"""Side-load timm==0.4.12 (Medical MAE pretrain pin from requirements.txt).

The medical_mae VisionTransformer subclass overrides forward_features(self, x),
matching the timm 0.4.x parent signature. Modern timm (1.x) calls
forward_features(x, attn_mask=..., is_causal=...) and breaks the override.

Same pattern as radharmony/evaluator/backbones/ark_plus.py: --no-deps install
into a side-by-side dir, pop modern timm from sys.modules, prepend the dir to
sys.path. The swap persists for the rest of this notebook.
"""

import os, sys, shutil, subprocess

MAE_DIR = "../../third_party_models/medical_mae"
LEGACY_TIMM_DIR = os.path.join(MAE_DIR, "timm-0412")

# 1. Clone the medical_mae repo if absent.
if not os.path.exists(MAE_DIR):
    subprocess.run(["git", "clone", "https://github.com/lambert-x/medical_mae",
                    MAE_DIR], check=True)

# 2. Side-install timm==0.4.12 if absent or contaminated with torchvision
#    (--no-deps is critical: prevents bundling an old torchvision that would
#    ABI-clash with the venv torch).
def _ensure_legacy_timm(legacy_dir: str) -> None:
    needs_install = (
        not os.path.exists(legacy_dir)
        or os.path.exists(os.path.join(legacy_dir, "torchvision"))
    )
    if needs_install:
        if os.path.exists(legacy_dir):
            shutil.rmtree(legacy_dir)
        subprocess.run(
            ["uv", "pip", "install",
             "--target", legacy_dir,
             "--python", sys.executable,
             "--no-deps",
             "timm==0.4.12"],
            check=True,
        )

_ensure_legacy_timm(LEGACY_TIMM_DIR)

# 3. Swap modern timm out of sys.modules, prepend legacy dir.
#    Saved modules are kept in case you want to restore them later
#    (sys.modules.update(_saved_timm); sys.path.remove(LEGACY_TIMM_DIR)).
_saved_timm = {k: sys.modules[k] for k in list(sys.modules)
               if k == "timm" or k.startswith("timm.")}
for k in _saved_timm:
    del sys.modules[k]
if LEGACY_TIMM_DIR not in sys.path:
    sys.path.insert(0, LEGACY_TIMM_DIR)

# 4. Add medical_mae to sys.path for the models_vit/util imports below.
if MAE_DIR not in sys.path:
    sys.path.insert(0, MAE_DIR)

import timm
print("timm version (legacy, scoped to notebook):", timm.__version__)
print("medical_mae files:", os.listdir(MAE_DIR)[:6], "...")


In [ ]:
import os

# Control which GPU(s) are visible to this notebook.
# Set CUDA_VISIBLE_DEVICES in your shell before launching Jupyter to override,
# e.g.:  CUDA_VISIBLE_DEVICES=2 jupyter lab
# Or change the default value below.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "6")
print(f"CUDA_VISIBLE_DEVICES = {os.environ['CUDA_VISIBLE_DEVICES']}")

## 1. Locate checkpoint

Download links are in the repo README table (Google Drive). Pick one:
- `vit_small_patch16_CXR_0.3M_mae_pretrain.pth` — ViT-S, 22M params
- `vit_base_patch16_CXR_0.5M_mae_pretrain.pth` — ViT-B, 86M params

In [ ]:
import os

# Set VARIANT to 'small' or 'base'
VARIANT = "base"

CKPT_NAMES = {
    # "small": "vit_small_patch16_CXR_0.3M_mae_pretrain.pth",
    "base":  "vit-b_CXR_0.5M_mae.pth",
}
EMBED_DIMS = {"base": 768}

CKPT_PATH = f"../../third_party_models/medical_mae/{CKPT_NAMES[VARIANT]}"

assert os.path.exists(CKPT_PATH), (
    f"Checkpoint not found at {CKPT_PATH}.\n"
    "Download from the Google Drive link in the lambert-x/medical_mae README."
)
print("Checkpoint found:", os.path.getsize(CKPT_PATH) / 1e6, "MB")
print("Expected embed dim:", EMBED_DIMS[VARIANT])

## 2. Load model

Medical MAE stores weights under the `model` key in the checkpoint with **flat
keys** (no `encoder.` prefix). The pretrained MAE model contains both encoder
and decoder weights — decoder keys, `mask_token`, and `decoder_*` are expected
unexpected keys when loading into a standalone ViT.

This mirrors the official loader in
[`main_linprobe_chestxray.py`](../../third_party_models/medical_mae/main_linprobe_chestxray.py):
flat keys, head dropped if shape-mismatched, `interpolate_pos_embed` for any
input-size change.


In [ ]:
import torch
from models_vit import vit_base_patch16
from util.pos_embed import interpolate_pos_embed

FACTORY = {"base": vit_base_patch16}

# Default Medical MAE eval path: CLS token (global_pool=False) uses the
# pretrained self.norm. With global_pool=True, fc_norm is freshly initialized.
model = FACTORY[VARIANT](num_classes=0, global_pool=False)

# Load checkpoint. Keys are flat (no encoder. prefix); decoder/mask_token are
# expected unexpected keys.
ckpt = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
print("Checkpoint top-level keys:", list(ckpt.keys()))
state = ckpt["model"]

# Drop head if shape-mismatched (mirrors main_linprobe_chestxray.py:211-214)
ref = model.state_dict()
for k in ("head.weight", "head.bias"):
    if k in state and k in ref and state[k].shape != ref[k].shape:
        del state[k]

# Interpolate pos-embed if input size differs from pretrained (no-op at 224)
interpolate_pos_embed(model, state)

missing, unexpected = model.load_state_dict(state, strict=False)
print("Missing:   ", missing)            # expected: head.weight, head.bias
print("Unexpected:", unexpected[:5], "...")  # decoder_*, mask_token, decoder_pred.*
model.eval()

# Keep `encoder_state` name for downstream cells that rebuild the CLS-token
# model (cell-14). It is just the same `state` we already loaded.
encoder_state = state
print(f"ViT-{VARIANT.upper()} loaded.")


## 3. Inspect checkpoint structure (diagnostic)

In [ ]:
all_keys = list(state.keys())
print("Total keys:", len(all_keys))
print("First 15:  ", all_keys[:15])
print("Last  10:  ", all_keys[-10:])

## 4. Image transform

Mirrors `Augmentation(normalize="chestx-ray").get_augmentation("full_224", "val")`
from [`util/dataloader_med.py`](../../third_party_models/medical_mae/util/dataloader_med.py):
Resize(256) → CenterCrop(224) → ToTensor → Normalize with **chestx-ray** stats
(`[0.5056]*3 / [0.252]*3`), not ImageNet.

For CXR sources the image is converted to 3-channel RGB up front (R==G==B for
true grayscale), matching `Image.open(file).convert("RGB")` in the official
`ChestX_ray14` loader.


In [ ]:
from torchvision import transforms
from PIL import Image

# Mirrors Augmentation(normalize="chestx-ray").get_augmentation("full_224", "val")
# from util/dataloader_med.py: Resize(256) -> CenterCrop(224) -> ToTensor ->
# Normalize(chestx-ray stats). Default torchvision Resize interpolation is
# BILINEAR (matches the official _full method).
INPUT_SIZE  = 224
RESIZE_SIZE = 256

def make_transform():
    return transforms.Compose([
        transforms.Lambda(lambda x: x.convert("RGB")),
        transforms.Resize(RESIZE_SIZE),
        transforms.CenterCrop(INPUT_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.5056, 0.5056, 0.5056],
            std =[0.252,  0.252,  0.252],
        ),
    ])

transform = make_transform()

def preprocess(path: str) -> torch.Tensor:
    return transform(Image.open(path)).unsqueeze(0)

dummy = torch.zeros(1, 3, INPUT_SIZE, INPUT_SIZE)
print("dummy input:", dummy.shape)


## 5. Extract embeddings

With `global_pool=False` and `num_classes=0`, `model(x)` returns the **CLS
token** `[B, D]` produced by the pretrained `self.norm`. Switching to
`global_pool=True` would return the mean of patch tokens passed through a
freshly-initialized `fc_norm` instead.


In [ ]:
with torch.no_grad():
    emb = model(dummy)   # [B, embed_dim] - CLS token (global_pool=False)

print("embedding shape:", emb.shape)   # [1, 768] for ViT-B
print("L2 norm:", emb.norm(dim=-1).item())


## 6. CLS token vs global avg pool

`model` (default, `global_pool=False`) returns the CLS token through the
pretrained `self.norm`. Rebuild with `global_pool=True` to compare against the
mean-pooled patch tokens normalized by the freshly-init `fc_norm`.


In [ ]:
# Rebuild with global_pool=True for the avg-pool comparison.
# `fc_norm.{weight,bias}` will be missing (freshly initialized).
model_pool = FACTORY[VARIANT](num_classes=0, global_pool=True)
model_pool.load_state_dict(encoder_state, strict=False)
model_pool.eval()

with torch.no_grad():
    emb_cls  = model(dummy)        # [B, D] - CLS token (pretrained norm)
    emb_pool = model_pool(dummy)   # [B, D] - mean(patches) -> fresh fc_norm

print("CLS token:    ", emb_cls.shape)
print("avg pool:     ", emb_pool.shape)
import torch.nn.functional as F
print("cosine(CLS, pool):",
      F.cosine_similarity(emb_cls, emb_pool).item())


## Segmentation mode - patch tokens

For dense prediction, capture all patch tokens from the final LayerNorm
(`model.norm`) via a forward hook. Since the default `model` is built with
`global_pool=False`, `self.norm` still exists and applies to the full token
sequence `[CLS, patch_1, ..., patch_N]`. For ViT-B/16 at 224x224:
**14x14** patch grid, 768-dim.


In [ ]:
import math

# Capture all tokens from the final transformer LayerNorm via a forward hook.
# `model` is global_pool=False, so model.norm applies to the full
# [CLS, patch_1, ..., patch_N] sequence.
all_tokens_out = {}

def _hook(module, inp, out):
    all_tokens_out["tokens"] = out   # [B, 1 + N_patches, D]

handle = model.norm.register_forward_hook(_hook)
with torch.no_grad():
    _ = model(dummy)
handle.remove()

all_tokens = all_tokens_out["tokens"]  # [B, 1 + N_patches, D]
patches    = all_tokens[:, 1:]         # [B, N_patches, D] - skip CLS

B, N, D = patches.shape
H = W = int(math.sqrt(N))   # 14 for 224px / 16px-patches (ViT-B)
assert H * W == N

spatial = patches.permute(0, 2, 1).reshape(B, D, H, W)
print(f"patch tokens:  {patches.shape}")   # [1, 196, 768]
print(f"spatial map:   {spatial.shape}")   # [1, 768, 14, 14]


## Notes for RadHarmony integration

```
model_call : None  (model(x) returns [B, D] with global_pool=False, num_classes=0)
pool       : None
embed_dim  : 768 (Base) / 384 (Small)
transform  : Resize(256) -> CenterCrop(224) -> ToTensor
             -> Normalize([0.5056]*3, [0.252]*3)   # chestx-ray stats
```

**Default eval path** (mirrors `main_linprobe_chestxray.py` defaults): CLS
token (`global_pool=False`) using the pretrained `self.norm`. `global_pool=True`
warm-starts `fc_norm` from default LayerNorm init and is intended for
fine-tuning, not zero-shot feature extraction.

**timm version conflict**: `models_vit.py` uses `timm==0.4.12` API
(`trunc_normal_` import path differs in newer timm). This may conflict with
EVA-X (0.9.0) and Ark+ (0.5.4). If side-loading is needed, mirror the
`make_ark_plus` pattern in [radharmony/evaluator/backbones/ark_plus.py](../../radharmony/evaluator/backbones/ark_plus.py).

**Checkpoint key format**: the released MAE pretrain checkpoint has **flat**
keys under `ckpt['model']` (no `encoder.` prefix); `decoder_*`, `mask_token`,
`decoder_pred.*` are expected unexpected keys. Drop `head.{weight,bias}` only
if shape-mismatched and call `interpolate_pos_embed` for any non-224 input.

**Segmentation mode** (`output_keys={"img", "mask"}`):
- Forward hook on `model.norm` (global_pool=False keeps `self.norm`)
- Patch tokens: `all_tokens[:, 1:]` -> `[B, 196, 768]`
- Spatial map: reshape to `[B, 768, 14, 14]` (224px / 16px-patches -> 14x14)


In [ ]:
CXR_PATH = "/path/to/cxr.jpg"

x = preprocess(CXR_PATH)
with torch.no_grad():
    emb = model(x)

print("embedding:", emb.shape)
print("L2 norm:  ", emb.norm(dim=-1).item())

## Notes for RadHarmony integration

```
model_call : None  (model(x) returns [B, D] with global_pool=False, num_classes=0)
pool       : None
embed_dim  : 768 (Base) / 384 (Small)
transform  : Resize(256) -> CenterCrop(224) -> ToTensor
             -> Normalize([0.5056]*3, [0.252]*3)   # chestx-ray stats
```

**Default eval path** (mirrors `main_linprobe_chestxray.py` defaults): CLS
token (`global_pool=False`) using the pretrained `self.norm`. `global_pool=True`
warm-starts `fc_norm` from default LayerNorm init and is intended for
fine-tuning, not zero-shot feature extraction.

**timm version conflict**: `models_vit.py` uses `timm==0.4.12` API
(`trunc_normal_` import path differs in newer timm). This may conflict with
EVA-X (0.9.0) and Ark+ (0.5.4). If side-loading is needed, mirror the
`make_ark_plus` pattern in [radharmony/evaluator/backbones/ark_plus.py](../../radharmony/evaluator/backbones/ark_plus.py).

**Checkpoint key format**: the released MAE pretrain checkpoint has **flat**
keys under `ckpt['model']` (no `encoder.` prefix); `decoder_*`, `mask_token`,
`decoder_pred.*` are expected unexpected keys. Drop `head.{weight,bias}` only
if shape-mismatched and call `interpolate_pos_embed` for any non-224 input.

**Segmentation mode** (`output_keys={"img", "mask"}`):
- Forward hook on `model.norm` (global_pool=False keeps `self.norm`)
- Patch tokens: `all_tokens[:, 1:]` -> `[B, 196, 768]`
- Spatial map: reshape to `[B, 768, 14, 14]` (224px / 16px-patches -> 14x14)
